In [1]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as et

In [2]:
# ---------- Goal of this script ----------
# The goal of this script is to load and combine the trip data from 2024 and 2025.


# ---------- What I will use this data for? ----------
# This data will be used for stohastic project. 

# Coded by: Artur Werys

In [3]:
# ---------- Paths ----------

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "Data"

DATA_2024_DIR = DATA_DIR / "2024"
DATA_2025_DIR = DATA_DIR / "2025"

STATIONS_FILE = DATA_DIR / "stations.xml"

In [4]:
# ---------- Function for loading CSV data ----------

def load_data(folder_path):
    files = list(folder_path.glob("*.csv"))
    dataframes = []

    for file in files:
        df = pd.read_csv(file)
        dataframes.append(df)

    combined_data = pd.concat(dataframes, ignore_index=True)

    return combined_data

In [5]:
# ---------- Loading data from choosen years ----------

print("--- Loading data from 2025... ---")

data_2025 = load_data(DATA_2025_DIR)

print(data_2025.head())
print("Number of trips in 2025:", len(data_2025))


print("--- Loading data from 2024... ---")

data_2024 = load_data(DATA_2024_DIR)

print(data_2024.head())
print("Number of trips in 2024:", len(data_2024))

--- Loading data from 2025... ---
      Number        Start date  Start station number  \
0  145666816  2025-01-14 23:59                  1043   
1  145666817  2025-01-14 23:59                300015   
2  145666818  2025-01-14 23:59                  1068   
3  145666819  2025-01-14 23:59                  1159   
4  145666812  2025-01-14 23:58                200048   

                      Start station          End date  End station number  \
0        Museum of London, Barbican  2025-01-15 00:13            200149.0   
1          Binfield Road, Stockwell  2025-01-15 00:04            300229.0   
2  Norton Folgate, Liverpool Street  2025-01-15 00:10              1051.0   
3         Berry Street, Clerkenwell  2025-01-15 00:10              1007.0   
4          Page Street, Westminster  2025-01-15 00:14              1058.0   

                        End station  Bike number  Bike model Total duration  \
0           Watney Street, Shadwell        55223     CLASSIC        14m 30s   
1       

In [6]:
# ---------- Combining datasets ----------

print("--- Combining data from 2024 and 2025... ---")

combined_trip_data = pd.concat(
    [data_2024, data_2025],
    ignore_index=True
)

--- Combining data from 2024 and 2025... ---


In [7]:
# ---------- Checking for missing values in combined data ----------

print("Missing values in each column:")
print(combined_trip_data.isna().sum())

len_before_dropping = len(combined_trip_data)

print("Dropping rows with missing data...")

combined_trip_data = combined_trip_data.dropna(
    subset=["End date", "End station number", "Bike number"]
)

len_after_dropping = len(combined_trip_data)

dropped_rows_count = len_before_dropping - len_after_dropping

print("Missing values in each column after dropping rows with missing data:")
print(combined_trip_data.isna().sum())

print(f"Number of dropped rows: {dropped_rows_count}")

Missing values in each column:
Number                    0
Start date                0
Start station number      0
Start station             0
End date                184
End station number      184
End station             184
Bike number               1
Bike model                0
Total duration          184
Total duration (ms)     184
dtype: int64
Dropping rows with missing data...
Missing values in each column after dropping rows with missing data:
Number                  0
Start date              0
Start station number    0
Start station           0
End date                0
End station number      0
End station             0
Bike number             0
Bike model              0
Total duration          0
Total duration (ms)     0
dtype: int64
Number of dropped rows: 185


In [8]:
# ---------- Loading station data from XML ----------
tree = et.parse(STATIONS_FILE)
root = tree.getroot()

stations_data = []

for station in root.findall("station"):

    station_data = [
        station.find("name").text.strip(),
        station.find("lat").text,
        station.find("long").text
    ]

    stations_data.append(station_data)

stations_df = pd.DataFrame(
    stations_data,
    columns=[
        "Station name",
        "Latitude",
        "Longitude"
    ]
)

print("--- Station data loaded from XML ---")
print(stations_df.head())
print("Number of stations:", len(stations_df))

--- Station data loaded from XML ---
                           Station name     Latitude     Longitude
0            River Street , Clerkenwell  51.52916347  -0.109970527
1        Phillimore Gardens, Kensington  51.49960695  -0.197574246
2  Christopher Street, Liverpool Street  51.52128377  -0.084605692
3       St. Chad's Street, King's Cross  51.53005939  -0.120973687
4         Sedding Street, Sloane Square     51.49313     -0.156876
Number of stations: 801


In [9]:
# ---------- Merging stations from trip data with geograpical data ----------
combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="Start station", right_on="Station name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "Latitude": "Start latitude",
    "Longitude": "Start longitude"
})

combined_trip_data = combined_trip_data.drop(columns=["Station name"])
combined_trip_data.head()

combined_trip_data = pd.merge(combined_trip_data, stations_df, left_on="End station", right_on="Station name", how ="left")
combined_trip_data = combined_trip_data.rename(columns={
    "Latitude": "End latitude",
    "Longitude": "End longitude"
})

combined_trip_data = combined_trip_data.drop(columns=["Station name"])
combined_trip_data.head()

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms),Start latitude,Start longitude,End latitude,End longitude
0,136666627,2024-01-14 23:59,1108,"North Wharf Road, Paddington",2024-01-15 00:06,3423.0,"Maida Vale, Maida Vale",53020.0,CLASSIC,6m 47s,407799.0,51.518623,-0.17660300000000007,51.529857,-0.18348604
1,136666625,2024-01-14 23:57,3447,"Gloucester Road (North), Kensington",2024-01-15 00:05,1214.0,"Kensington Olympia Station, Olympia",54559.0,CLASSIC,8m 1s,481276.0,51.49792478,-0.183834706,51.49815779,-0.209494128
2,136666626,2024-01-14 23:57,1090,"Warren Street Station, Euston",2024-01-15 00:02,200005.0,"New Cavendish Street, Marylebone",52133.0,CLASSIC,5m 11s,311788.0,51.52443845,-0.138019439,51.519167,-0.147983
3,136666622,2024-01-14 23:56,200058,"Northdown Street, King's Cross",2024-01-15 00:06,2687.0,"Brick Lane Market, Shoreditch",60341.0,PBSC_EBIKE,10m 14s,614030.0,51.531066,-0.11934,51.52261762,-0.071653961
4,136666623,2024-01-14 23:56,1052,"Soho Square , Soho",2024-01-15 00:23,1150.0,"Queensway, Kensington Gardens",55212.0,CLASSIC,27m 1s,1621583.0,51.51563144,-0.132328837,51.51031,-0.18740235


In [10]:
# ---------- Deleting trips without geographical data ----------
final_trip_data = combined_trip_data.dropna(subset=["Start latitude", "Start longitude", "End latitude", "End longitude"])

# ---------- How many trips were deleted ----------
print(f"Percentage of trips deleted: {((len(combined_trip_data) - len(final_trip_data)) / len(combined_trip_data) * 100):.2f}%")


Percentage of trips deleted: 2.23%
